# Генерация синтетического датасета поведенческих признаков

Генерирует данные для модели классификации аномального поведения кандидата:

- признаки совпадают с полями, которые уже есть в таблице `behavior_features`;
- целевая переменная `is_anomaly` для обучения модели;
- генерация воспроизводима за счет фиксированного `RANDOM_SEED`.


CSV будет сохранен в `ml/data/synthetic_behavior_dataset.csv`.

In [1]:
# pathlib для работы с путями к файлам
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Общие настройки

`RANDOM_SEED` фиксирует случайность для воспроизводимости:

In [2]:
RANDOM_SEED = 42 # для воспроизводимости результатов
ROWS_COUNT = 5000 # количество строк в итоговом датасете 
ANOMALY_SHARE = 0.30 # доля аномальных сессий в итоговом датасете (30% аномальных сессий и 70% нормальных)
# 50/50 слишком искусственно - нормальных ответов обычно больше, чем подозрительны
# если сделать менее 10%, то будет сложно обучить модель, так как аномальных примеров будет мало
# 30% удобная эвристика, которая позволяет модели учиться на достаточном количестве аномальных примеров, но при этом сохраняет реалистичное соотношение между нормальными и аномальными сессиями

# фиксируем сид для генератора случайных чисел, чтобы результаты были воспроизводимыми
rng = np.random.default_rng(RANDOM_SEED)

# Запускаем notebook из корня проекта FINAL.
# Тогда итоговый CSV сохраняется в папку ml/data.
DATA_DIR = Path("ml/data")
DATA_DIR.mkdir(parents=True, exist_ok=True) # создаем папку для данных, если ее еще нет

OUTPUT_PATH = DATA_DIR / "synthetic_behavior_dataset.csv" # итоговый путь к CSV файлу с синтетическим датасетом

## 2. Признаки, совпадающие с backend и базой

Эти признаки соответствуют:
- классу `BehaviorMetrics` в `backend/behavior.py`;
- колонкам таблицы `behavior_features` в `postgres/init/01_schema.sql`.

In [3]:
# список названий колонок с признаками, которые будут использоваться для обучения модели
FEATURE_COLUMNS = [
    "response_time_sec",       # общее время ответа в секундах
    "tab_switch_count",        # сколько раз кандидат уходил со страницы или вкладка становилась скрытой
    "total_hidden_time_sec",   # суммарное время, когда страница была скрыта
    "hidden_time_ratio",       # доля времени вне страницы: total_hidden_time_sec / response_time_sec
    "paste_count",             # количество вставок текста из буфера обмена
    "pasted_chars_total",      # сколько символов было вставлено через paste
    "paste_ratio",             # доля вставленного текста: pasted_chars_total / answer_length_chars
    "input_event_count",       # количество событий изменения ответа (вставки, удаления, правки текста)
    "delete_count",            # количество удалений и правок текста
    "max_pause_sec",           # максимальная пауза между изменениями ответа (в секундах)
    "max_input_burst_chars",   # максимальный резкий прирост длины ответа за одно изменение (например, если кандидат внезапно написал длинный абзац, это может быть подозрительно)
    "copy_count",              # количество копирований текста со страницы (может указывать на то, что кандидат копирует информацию и использует в других местах)
    "answer_length_chars",     # итоговая длина ответа в символах 
]

TARGET_COLUMN = "is_anomaly" # название колонки с целевой переменной, которая указывает, является ли сессия аномальной (1) или нормальной (0)
DATASET_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN] # полный список колонок в итоговом датасете (признаки + целевая переменная)

# словарь с описаниями каждого признака для удобства понимания, что означает каждый признак и как он может быть полезен для модели
FEATURE_DESCRIPTIONS = {
    "response_time_sec": "Общее время ответа в секундах.",
    "tab_switch_count": "Сколько раз кандидат уходил со страницы или вкладка становилась скрытой.",
    "total_hidden_time_sec": "Суммарное время, когда страница была скрыта.",
    "hidden_time_ratio": "Доля времени вне страницы: total_hidden_time_sec / response_time_sec.",
    "paste_count": "Количество вставок текста из буфера обмена.",
    "pasted_chars_total": "Сколько символов было вставлено через paste.",
    "paste_ratio": "Доля вставленного текста: pasted_chars_total / answer_length_chars.",
    "input_event_count": "Количество событий изменения ответа.",
    "delete_count": "Количество удалений и правок текста.",
    "max_pause_sec": "Максимальная пауза между изменениями ответа.",
    "max_input_burst_chars": "Максимальный резкий прирост длины ответа за одно изменение.",
    "copy_count": "Количество копирований текста со страницы.",
    "answer_length_chars": "Итоговая длина ответа в символах.",
}

for feature_name, description in FEATURE_DESCRIPTIONS.items():
    print(f"{feature_name}: {description}")

response_time_sec: Общее время ответа в секундах.
tab_switch_count: Сколько раз кандидат уходил со страницы или вкладка становилась скрытой.
total_hidden_time_sec: Суммарное время, когда страница была скрыта.
hidden_time_ratio: Доля времени вне страницы: total_hidden_time_sec / response_time_sec.
paste_count: Количество вставок текста из буфера обмена.
pasted_chars_total: Сколько символов было вставлено через paste.
paste_ratio: Доля вставленного текста: pasted_chars_total / answer_length_chars.
input_event_count: Количество событий изменения ответа.
delete_count: Количество удалений и правок текста.
max_pause_sec: Максимальная пауза между изменениями ответа.
max_input_burst_chars: Максимальный резкий прирост длины ответа за одно изменение.
copy_count: Количество копирований текста со страницы.
answer_length_chars: Итоговая длина ответа в символах.


## 3. Вспомогательные функции

Эти функции нужны, чтобы значения всегда укладывались в ограничения базы данных:
- счетчики не могут быть отрицательными;
- доли должны быть от `0` до `1`;

In [4]:
# функция для ограничения числовых значений в определенном диапазоне, чтобы избежать некорректных данных (например, отрицательного времени ответа)
def clamp(value: float, lower: float, upper: float) -> float:
    """Ограничивает число диапазоном от lower до upper."""

    return min(max(value, lower), upper) # если value меньше lower, возвращаем lower; если value больше upper, возвращаем upper; иначе возвращаем value

# функция для округления числовых значений до целого числа и обеспечения того, что они не будут меньше определенного нижнего предела (например, количество вставок не может быть отрицательным)
def as_int(value: float, lower: int = 0) -> int:
    """Округляет число до int и не дает ему стать меньше lower."""

    return max(int(round(value)), lower) # округляем value до ближайшего целого числа, а затем гарантируем, что результат не будет меньше lower

# функция для безопасного вычисления доли между двумя числами, которая гарантирует, что результат будет в диапазоне от 0 до 1 (например, доля скрытого времени не может быть отрицательной или больше 1)
def safe_ratio(numerator: float, denominator: float) -> float:
    """Считает долю и гарантирует диапазон от 0 до 1."""

    # numerator - числитель (например, время вне страницы), denominator - знаменатель (например, общее время ответа)
    if denominator <= 0: # чтобы избежать деления на ноль или отрицательное время, возвращаем 0.0, если знаменатель не положительный
        return 0.0
    return round(clamp(numerator / denominator, 0.0, 1.0), 6) # вычисляем долю, ограничиваем ее диапазоном от 0 до 1 и округляем до 6 знаков после запятой

# функция для приведения строки с признаками к формату, который будет сохранен в итоговом CSV файле. Она выполняет очистку и нормализацию данных, а также добавляет целевую переменную is_anomaly.
def finalize_row(row: dict, is_anomaly: int) -> dict:
    """Приводит строку к формату, который будет сохранен в CSV."""

    response_time_sec = round(float(row["response_time_sec"]), 3) # общее время ответа в секундах, округленное до 3 знаков после запятой
    answer_length_chars = as_int(row["answer_length_chars"], lower=1) # итоговая длина ответа в символах, округленная до целого числа и гарантированно не меньше 1 (чтобы избежать деления на ноль при вычислении paste_ratio)

    # ограничиваем total_hidden_time_sec диапазоном от 0 до response_time_sec, так как время вне страницы не может быть отрицательным или больше общего времени ответа
    total_hidden_time_sec = round(
        clamp(float(row["total_hidden_time_sec"]), 0.0, response_time_sec),
        3,
    )
    # ограничиваем pasted_chars_total диапазоном от 0 до answer_length_chars, так как количество вставленных символов не может быть отрицательным или больше общей длины ответа
    pasted_chars_total = as_int(
        clamp(float(row["pasted_chars_total"]), 0.0, answer_length_chars),
        lower=0,
    )

    # формируем итоговую строку с признаками и целевой переменной, которая будет сохранена в CSV файле. 
    # Все числовые значения очищены и нормализованы, а также добавлена колонка is_anomaly
    return {
        "response_time_sec": response_time_sec,
        "tab_switch_count": as_int(row["tab_switch_count"]),
        "total_hidden_time_sec": total_hidden_time_sec,
        "hidden_time_ratio": safe_ratio(total_hidden_time_sec, response_time_sec),
        "paste_count": as_int(row["paste_count"]),
        "pasted_chars_total": pasted_chars_total,
        "paste_ratio": safe_ratio(pasted_chars_total, answer_length_chars),
        "input_event_count": as_int(row["input_event_count"], lower=1),
        "delete_count": as_int(row["delete_count"]),
        "max_pause_sec": round(max(float(row["max_pause_sec"]), 0.0), 3),
        "max_input_burst_chars": as_int(row["max_input_burst_chars"]),
        "copy_count": as_int(row["copy_count"]),
        "answer_length_chars": answer_length_chars,
        "is_anomaly": int(is_anomaly),
    }

## 4. Нормальное поведение (честное)

Нормальный сценарий не означает идеальное поведение. Кандидат может:
- иногда переключить вкладку;
- сделать небольшую вставку;
- исправлять ответ;
- делать паузы.

Но у нормального поведения обычно нет сильных сигналов сразу по нескольким
признакам: высокая доля вставленного текста, много времени вне страницы,
много переключений вкладок.

In [5]:
def generate_normal_row() -> dict:
    """Генерирует одну строку поведения без аномалии."""

    # Длина ответа: чаще средняя, иногда короткая или длинная.
    # Используем lognormal, потому что ответы не распределены равномерно:
    # чаще встречаются средние ответы, реже очень короткие или очень длинные.
    # mean=np.log(350) - типичная длина ответа около 350 символов.
    # sigma=0.55 задает разброс.
    # clamp ограничивает длину от 50 до 1600 символов, чтобы не получить нереалистично маленькие или огромные ответы
    answer_length_chars = clamp(
        rng.lognormal(mean=np.log(350), sigma=0.55),
        50,
        1600,
    )

    # Время ответа связано с длиной текста: длинный ответ обычно требует больше времени.
    # Для нормального поведения считаем, что кандидат набирает примерно от 1.2 до 3.5 символов в секунду.
    typing_speed_chars_per_sec = rng.uniform(1.2, 3.5)
    # answer_length_chars / typing_speed_chars_per_sec дает базовое время набора.
    # rng.normal(35, 20) добавляет время на чтение вопроса, размышления и правки.
    # clamp ограничивает итоговое время от 35 до 1200 секунд (чтобы исключить нереалистично короткие или длинные времена ответа)
    response_time_sec = clamp(
        answer_length_chars / typing_speed_chars_per_sec + rng.normal(35, 20),
        35,
        1200,
    )

    # Большинство нормальных кандидатов не уходят со страницы или делают это редко.
    # Для нормального кандидата чаще всего это 0.
    # Иногда может быть 1 переключение, редко 2.
    # Это не обязательно нарушение: кандидат мог случайно переключиться, открыть календарь, мессенджер, окно браузера и т.д.
    tab_switch_count = rng.choice([0, 1, 2], p=[0.78, 0.18, 0.04]) # 78% случаев - 0 переключений, 18% - 1 переключение, 4% - 2 переключения
    
    # Если кандидат не уходил со страницы, доля скрытого времени равна 0.
    # Если уходил, генерируем небольшую долю скрытого времени.
    # beta(1.2, 12) для дает значения ближе к нулю, то есть чаще короткие уходы. 
    # бета-распределение хорошо моделирует долю скрытого времени, так как оно ограничено между 0 и 1 и может быть скошенным в сторону меньших значений (чаще короткие уходы).
    # Умножаем на 0.45 и затем ограничиваем диапазон 0.005..0.20. 
    # Сжимаем максимум до 45% и ограничиваем, чтобы не было нереалистично больших долей скрытого времени.
    # То есть для нормального поведения страница скрыта максимум около 20% времени.
    hidden_time_ratio = 0.0 if tab_switch_count == 0 else clamp(rng.beta(1.2, 12) * 0.45, 0.005, 0.20)
    # Переводим долю скрытого времени в секунды.
    total_hidden_time_sec = response_time_sec * hidden_time_ratio

    # Небольшие вставки допустимы: например, кандидат вставил короткий SQL-фрагмент.
    # В норме чаще всего вставок нет.
    # 0 вставок - 80% случаев, 1 вставка - 18%, 2 вставки - 2%.
    paste_count = rng.choice([0, 1, 2], p=[0.80, 0.18, 0.02])
    # Если вставок нет, доля вставленного текста равна 0.
    # Если вставка была, генерируем небольшую долю вставленного текста.
    # Верхняя граница 0.25 означает, что в нормальном сценарии
    # вставлено не больше 25% всего ответа.
    paste_ratio = 0.0 if paste_count == 0 else clamp(rng.beta(1, 8) * 0.35, 0.01, 0.25)
    # Переводим долю вставленного текста в количество символов.
    pasted_chars_total = answer_length_chars * paste_ratio

    # input_event_count зависит от длины ответа: чем длиннее ответ, тем больше изменений.
    # Делим длину ответа на случайное число от 5 до 14
    # это имитирует, что одно событие ввода может добавлять несколько символов.
    # rng.normal(4, 2) добавляет небольшой шум.
    input_event_count = answer_length_chars / rng.uniform(5, 14) + rng.normal(4, 2)

    # Нормальный ответ обычно содержит правки и удаления.
    # poisson(answer_length_chars / 140) связывает число правок с длиной ответа:
    # чем длиннее ответ, тем больше ожидаемых исправлений.
    # rng.integers(0, 4) добавляет небольшой случайный разброс.
    # ответ 140 символов  -> примерно 1 правка
    # ответ 280 символов  -> примерно 2 правки
    # ответ 560 символов  -> примерно 4 правки
    # rng.poisson(...) добавляет естественную случайность: у одного кандидата будет 2 правки, у другого 5, даже при похожей длине ответа
    delete_count = rng.poisson(answer_length_chars / 140) + rng.integers(0, 4)

    # Паузы есть почти всегда: кандидат думает, читает вопрос, исправляет текст.
    # lognormal дает чаще умеренные паузы и редко длинные.
    # Типичная пауза около 12 секунд.
    # Ограничиваем диапазон 2..75 секунд.
    max_pause_sec = clamp(rng.lognormal(mean=np.log(12), sigma=0.65), 2, 75)

    # максимальный резкий прирост текста за одно изменение.
    # Если кандидат печатает руками, такой прирост обычно небольшой.
    # Типичное значение около 18 символов, верхняя граница 120.
    max_input_burst_chars = clamp(rng.lognormal(mean=np.log(18), sigma=0.65), 2, 120)
    
    # Если была вставка, то максимальный прирост может быть больше
    # Но для нормального поведения считаем, что вставленный фрагмент небольшой
    # берем только 25-75% от вставленных символов как возможный максимум резкого прироста
    if paste_count > 0:
        max_input_burst_chars = max(max_input_burst_chars, pasted_chars_total * rng.uniform(0.25, 0.75))

    # В нормальном сценарии копирование редкое:
    # 92% случаев - 0 копирований,
    # 8% случаев - 1 копирование.
    copy_count = rng.choice([0, 1], p=[0.92, 0.08])

    # Собираем все признаки в словарь и передаем в finalize_row
    return finalize_row(
        {
            "response_time_sec": response_time_sec,
            "tab_switch_count": tab_switch_count,
            "total_hidden_time_sec": total_hidden_time_sec,
            "paste_count": paste_count,
            "pasted_chars_total": pasted_chars_total,
            "input_event_count": input_event_count,
            "delete_count": delete_count,
            "max_pause_sec": max_pause_sec,
            "max_input_burst_chars": max_input_burst_chars,
            "copy_count": copy_count,
            "answer_length_chars": answer_length_chars,
        },
        is_anomaly=0, # для нормального поведения is_anomaly = 0
    )

## 5. Аномальное поведение

Аномальный класс строится не по одному признаку, а по нескольким сценариям:

1. `copy_paste` - большая часть ответа вставлена разом.
2. `external_help` - кандидат часто уходит со страницы и долго отсутствует.
3. `mixed` - умеренные признаки сразу по нескольким направлениям.

In [6]:
# функция для генерации строки с аномалией. 
# моделирует поведение, которое может указывать на использование запрещенных инструментов или помощь извне
# чрезмерное копирование, большое количество вставок, длинные периоды скрытого времени и т.д.
def generate_anomaly_row() -> dict:
    """Генерирует одну строку поведения с аномалией."""

    # Сценарии аномального поведения могут быть разными.
    # Например, "copy_paste" - кандидат активно копирует и вставляет текст -  45% случаев,
    # "external_help" - кандидат часто уходит со страницы и получает помощь извне - 35% случаев,
    # "mixed" - сочетание разных аномальных паттернов - 20% случаев.
    # делаем аномальный класс неоднородным, чтобы модель училась распознавать разные типы подозрительного поведения, а не один шаблон.
    scenario = rng.choice(["copy_paste", "external_help", "mixed"], p=[0.45, 0.35, 0.20])

    # Генерируем длину ответа
    # Для аномальных сценариев типичная длина чуть больше, чем у нормального ответа: около 430 символов вместо 350.
    # вставленные/сгенерированные ответы часто бывают более развернутыми
    # Ограничиваем длину диапазоном 60..2200 символов, чтобы не получить
    # слишком маленькие или нереалистично огромные ответы
    answer_length_chars = clamp(
        rng.lognormal(mean=np.log(430), sigma=0.65),
        60,
        2200,
    )


    if scenario == "copy_paste":
        # кандидат вставляет значительную часть ответа из буфера обмена
        # rng.integers(1, 5) дает целые числа от 1 до 4
        # в этом сценарии минимум одна вставка обязательна
        paste_count = rng.integers(1, 5)
        # Генерируем долю вставленного текста.
        # От 45% до 98% ответа может быть вставлено
        paste_ratio = rng.uniform(0.45, 0.98)
        # Переводим долю вставленного текста в количество символов
        pasted_chars_total = answer_length_chars * paste_ratio

        # Время ответа в copy_paste-сценарии обычно меньше - большая часть текста не набирается вручную
        # Делим длину ответа на высокую "скорость" 5..13 символов/сек,
        # затем добавляем немного времени на чтение, вставку и отправку
        # Ограничиваем итоговое время 20..450 секунд.
        response_time_sec = clamp(
            answer_length_chars / rng.uniform(5.0, 13.0) + rng.normal(25, 12),
            20,
            450,
        )
        
        # Переключения вкладок могут быть, но не обязательны:
        # при copy_paste кандидат мог заранее иметь текст в буфере обмена
        tab_switch_count = rng.choice([0, 1, 2, 3], p=[0.25, 0.35, 0.25, 0.15])
        # Если вкладки не переключались, скрытого времени нет.
        # Если переключения были, скрытое время может составлять от 3% до 35%.
        hidden_time_ratio = 0.0 if tab_switch_count == 0 else rng.uniform(0.03, 0.35)
        # Количество событий ввода при copy_paste меньше, чем при ручном наборе
        # большой кусок текста появляется сразу
        input_event_count = answer_length_chars / rng.uniform(30, 90) + rng.integers(1, 8)
        # Правок немного: если ответ вставлен готовым, кандидат меньше удаляет
        # и переписывает текст.
        delete_count = rng.poisson(2)
        # Максимальная пауза может быть умеренной:
        # кандидат мог ждать, копировать, читать внешний ответ.
        max_pause_sec = clamp(rng.lognormal(mean=np.log(18), sigma=0.8), 3, 180)
        # Максимальный прирост ввода большой, потому что вставленный текст появляется почти сразу.
        # Берем от 55% до 100% вставленного текста
        max_input_burst_chars = pasted_chars_total * rng.uniform(0.55, 1.0)
        # Копирование в этом сценарии возможно чаще
        copy_count = rng.choice([0, 1, 2, 3], p=[0.30, 0.35, 0.25, 0.10])

    elif scenario == "external_help":
        # кандидат часто уходит со страницы, вероятно, ищет информацию
        # в другом окне, мессенджере, поиске или получает помощь извне
        # Вставка текста здесь не обязательна
        # Может быть 0, 1 или 2 вставки
        paste_count = rng.choice([0, 1, 2], p=[0.50, 0.35, 0.15])
        # Если вставок нет, доля вставленного текста 0
        # Если есть, она умеренная: от 5% до 40%
        paste_ratio = 0.0 if paste_count == 0 else rng.uniform(0.05, 0.40)
        pasted_chars_total = answer_length_chars * paste_ratio

        # Время ответа обычно большое
        # кандидат уходит со страницы, думает, ищет или ждет подсказку.
        # Делим длину ответа на медленную скорость 0.7..2.0 символа/сек
        # и добавляем большую задержку normal(180, 80)
        response_time_sec = clamp(
            answer_length_chars / rng.uniform(0.7, 2.0) + rng.normal(180, 80),
            180,
            1800,
        )
        
        # Частые переключения вкладок
        # rng.integers(3, 15) дает от 3 до 14 переключений
        tab_switch_count = rng.integers(3, 15)
        
        # Доля времени вне страницы высокая: от 25% до 85%.
        hidden_time_ratio = rng.uniform(0.25, 0.85)
        
        # Количество событий ввода похоже на ручной ввод
        # кандидат мог получать помощь, но потом сам набирал/правил ответ
        input_event_count = answer_length_chars / rng.uniform(6, 18) + rng.normal(3, 2)
        
        # Количество правок связано с длиной ответа
        # Чем длиннее ответ, тем больше ожидаемых исправлений
        delete_count = rng.poisson(answer_length_chars / 180) + rng.integers(0, 5)
        
        # Максимальная пауза большая
        # кандидат мог ждать ответ, искать информацию или переключаться между источниками
        max_pause_sec = clamp(rng.lognormal(mean=np.log(90), sigma=0.75), 25, 600)
        
        # Максимальный прирост ввода может быть обычным или связанным со вставкой.
        # Поэтому берем максимум из:
        # случайного ручного прироста, который может быть умеренным (lognormal с типичным значением около 25 символов);
        # части вставленного текста.
        max_input_burst_chars = max(
            rng.lognormal(mean=np.log(25), sigma=0.8),
            pasted_chars_total * rng.uniform(0.3, 0.9),
        )
        
        # Копирование возможно чаще, потому что кандидат может копировать
        # информацию со страницы или из других источников, чтобы использовать в ответе.
        copy_count = rng.choice([0, 1, 2, 3, 4], p=[0.20, 0.25, 0.25, 0.20, 0.10])

    else:
        # смешанное подозрительное поведение
        # Здесь нет одного экстремального признака, но есть сочетание нескольких
        # вставки почти всегда есть: от 1 до 3
        paste_count = rng.integers(1, 4)
        # Доля вставленного текста средняя или высокая: от 20% до 65%
        paste_ratio = rng.uniform(0.20, 0.65)
        pasted_chars_total = answer_length_chars * paste_ratio

        # Время ответа среднее:
        # быстрее, чем при external_help, но не настолько быстро,
        # как при чистом copy_paste.
        response_time_sec = clamp(
            answer_length_chars / rng.uniform(2.5, 7.0) + rng.normal(60, 35),
            45,
            900,
        )
        
        # Несколько переключений вкладок: от 2 до 7.
        tab_switch_count = rng.integers(2, 8)
        # Доля скрытого времени умеренно высокая: от 12% до 55%.
        hidden_time_ratio = rng.uniform(0.12, 0.55)
        # Событий ввода меньше, чем при полностью ручном наборе,
        # потому что часть текста вставляется.
        input_event_count = answer_length_chars / rng.uniform(12, 45) + rng.integers(2, 10)
        
        # Правок немного, потому что часть ответа могла быть уже готовой
        delete_count = rng.poisson(3)
        # Максимальная пауза выше нормы, но обычно ниже, чем при external_help
        max_pause_sec = clamp(rng.lognormal(mean=np.log(45), sigma=0.75), 8, 300)
        max_input_burst_chars = max(
            rng.lognormal(mean=np.log(70), sigma=0.75),
            pasted_chars_total * rng.uniform(0.4, 0.95),
        )
        
        # Копирование встречается чаще, чем в нормальном сценарии
        copy_count = rng.choice([0, 1, 2, 3], p=[0.15, 0.35, 0.30, 0.20])

    return finalize_row(
        {
            "response_time_sec": response_time_sec,
            "tab_switch_count": tab_switch_count,
            "total_hidden_time_sec": response_time_sec * hidden_time_ratio,
            "paste_count": paste_count,
            "pasted_chars_total": pasted_chars_total,
            "input_event_count": input_event_count,
            "delete_count": delete_count,
            "max_pause_sec": max_pause_sec,
            "max_input_burst_chars": max_input_burst_chars,
            "copy_count": copy_count,
            "answer_length_chars": answer_length_chars,
        },
        is_anomaly=1, # для аномального поведения is_anomaly = 1
    )

## 6. Сборка датасета

Сначала создаем точное количество нормальных и аномальных строк, затем
перемешиваем. Так `ANOMALY_SHARE` получается контролируемым, а не случайно
плавающим от запуска к запуску.

In [7]:
def generate_dataset(rows_count: int, anomaly_share: float) -> pd.DataFrame:
    """Генерирует полный датасет."""

    # Вычисляем количество аномальных и нормальных строк на основе общего количества строк и доли аномалий.
    anomaly_count = int(round(rows_count * anomaly_share))
    # остальные строки будут нормальными
    normal_count = rows_count - anomaly_count

    # Генерируем нормальные строки и аномальные строки, а затем объединяем их в один список.
    rows = [generate_normal_row() for _ in range(normal_count)]
    rows += [generate_anomaly_row() for _ in range(anomaly_count)]

    # Создаем DataFrame из списка строк и задаем названия колонок.
    # Затем перемешиваем строки в DataFrame, чтобы аномальные и нормальные примеры были перемешаны, а не сгруппированы по порядку.
    df = pd.DataFrame(rows, columns=DATASET_COLUMNS)
    df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True) # frac=1 означает, что мы берем 100% строк, random_state фиксирует порядок перемешивания для воспроизводимости, reset_index(drop=True) сбрасывает индекс после перемешивания
    return df

# Генерируем датасет с заданным количеством строк и долей аномалий, а затем выводим первые несколько строк для проверки.
dataset = generate_dataset(ROWS_COUNT, ANOMALY_SHARE)
dataset.head()

,response_time_sec,tab_switch_count,total_hidden_time_sec,hidden_time_ratio,paste_count,pasted_chars_total,paste_ratio,input_event_count,delete_count,max_pause_sec,max_input_burst_chars,copy_count,answer_length_chars,is_anomaly
0,232.753,0,0.000,0.000000,0,0,0.0,56,10,10.504,44,0,617,0
1,259.177,1,26.370,0.101745,0,0,0.0,50,3,5.935,16,0,276,0
2,213.754,0,0.000,0.000000,0,0,0.0,42,6,24.093,58,0,471,0
3,637.918,1,48.717,0.076369,0,0,0.0,250,12,6.551,27,0,1371,0
4,297.998,0,0.000,0.000000,0,0,0.0,68,6,6.620,38,0,632,0


## 7. Проверка качества сгенерированных данных

Проверяем корректность самого датасета:
- нет лишних колонок;
- нет пропусков;
- нет отрицательных значений;
- доли находятся в диапазоне `0..1`.

In [8]:
def validate_dataset(df: pd.DataFrame) -> None:
    """Падает с ошибкой, если датасет нарушает ожидаемую структуру."""

    if list(df.columns) != DATASET_COLUMNS:
        raise ValueError(f"Неожиданные колонки: {list(df.columns)}")

    if df.isna().any().any():
        raise ValueError("В датасете есть пропуски.")

    non_negative_columns = [
        "response_time_sec",
        "tab_switch_count",
        "total_hidden_time_sec",
        "paste_count",
        "pasted_chars_total",
        "input_event_count",
        "delete_count",
        "max_pause_sec",
        "max_input_burst_chars",
        "copy_count",
        "answer_length_chars",
    ]
    if (df[non_negative_columns] < 0).any().any():
        raise ValueError("В датасете есть отрицательные значения.")

    if not df["hidden_time_ratio"].between(0, 1).all():
        raise ValueError("hidden_time_ratio вышел за диапазон 0..1.")

    if not df["paste_ratio"].between(0, 1).all():
        raise ValueError("paste_ratio вышел за диапазон 0..1.")

    expected_hidden_ratio = (df["total_hidden_time_sec"] / df["response_time_sec"]).clip(0, 1).round(6)
    expected_paste_ratio = (df["pasted_chars_total"] / df["answer_length_chars"]).clip(0, 1).round(6)

    if not np.allclose(df["hidden_time_ratio"], expected_hidden_ratio, atol=0.00001):
        raise ValueError("hidden_time_ratio не совпадает с total_hidden_time_sec / response_time_sec.")

    if not np.allclose(df["paste_ratio"], expected_paste_ratio, atol=0.00001):
        raise ValueError("paste_ratio не совпадает с pasted_chars_total / answer_length_chars.")

    if set(df[TARGET_COLUMN].unique()) != {0, 1}:
        raise ValueError("is_anomaly должен содержать только 0 и 1.")


validate_dataset(dataset)
print("Датасет прошел проверку.")

Датасет прошел проверку.


## 8. Быстрая аналитика

Эти таблицы помогают глазами проверить, что нормальное и аномальное поведение
действительно отличаются по смысловым признакам.

In [ ]:
print("Размер датасета:", dataset.shape)
print("\nРаспределение классов:")
print(dataset[TARGET_COLUMN].value_counts(normalize=True).rename("share").round(3)) # share - доля каждого класса в датасете, normalize=True означает, что мы хотим получить долю, а не абсолютные количества, round(3) округляет результат до 3 знаков после запятой

print("\nСредние значения признаков по классам:")
summary = dataset.groupby(TARGET_COLUMN)[FEATURE_COLUMNS].mean().round(3)
print(summary)

Размер датасета: (5000, 14)

Распределение классов:
is_anomaly
0    0.7
1    0.3
Name: share, dtype: float64

Средние значения признаков по классам:
            response_time_sec  tab_switch_count  total_hidden_time_sec  \
is_anomaly                                                               
0                     221.945             0.256                  1.969   
1                     292.088             4.467                136.661   

            hidden_time_ratio  paste_count  pasted_chars_total  paste_ratio  \
is_anomaly                                                                    
0                       0.009        0.224               3.239        0.008   
1                       0.327        1.751             247.556        0.445   

            input_event_count  delete_count  max_pause_sec  \
is_anomaly                                                   
0                      50.173         4.384         15.004   
1                      30.043         3.331        

## 9. Сохранение CSV

Итоговый файл сохраняется в:

`ml/data/synthetic_behavior_dataset.csv`

Этот файл потом можно использовать в следующем notebook/script для обучения
моделей классификации.

In [10]:
dataset.to_csv(OUTPUT_PATH, index=False)
print(f"CSV сохранен: {OUTPUT_PATH}")

CSV сохранен: ml/data/synthetic_behavior_dataset.csv
